# Content Pipeline Debug Notebook

This notebook breaks the content pipeline into step-by-step stages so we can debug scraping coverage and backend data quality incrementally.

Recommended loop:
1. Run setup and load module/resource metadata.
2. Run Stage 1 scraping on a subset (`MAX_URLS`) until quality checks look good.
3. Run Stage 2/3 to inspect source docs, chunks, and bucket assignment quality.
4. Enable Stage 4 only when you want to run LLM synthesis (`RUN_LLM_STAGES=True`).


In [1]:
from __future__ import annotations

import json
import logging
import os
import re
import sys
import textwrap
import time
from collections import Counter
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Iterable
from urllib.parse import urlparse


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "build_study_data.py").exists() and (candidate / "pipeline").exists():
            return candidate
    raise RuntimeError("Run this notebook from inside the study-app repository.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
PIPELINE_DIR = REPO_ROOT / "pipeline"
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))


LOG_LEVEL = os.getenv("PIPELINE_DEBUG_LOG_LEVEL", "INFO").upper()
logging.basicConfig(
    level=getattr(logging, LOG_LEVEL, logging.INFO),
    format="%(asctime)s.%(msecs)03d %(levelname)s %(name)s: %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("content-pipeline-debug")
log.setLevel(getattr(logging, LOG_LEVEL, logging.INFO))


@dataclass
class Timer:
    name: str
    start: float = field(default_factory=time.perf_counter)

    def ms(self) -> int:
        return int((time.perf_counter() - self.start) * 1000)


def preview_text(value: Any, *, max_chars: int = 260) -> str:
    if value is None:
        return ""
    if isinstance(value, (dict, list, tuple, set)):
        try:
            value = json.dumps(value, ensure_ascii=False)  # type: ignore[arg-type]
        except Exception:
            value = str(value)
    text = str(value)
    text = re.sub(r"\s+", " ", text).strip()
    if len(text) <= max_chars:
        return text
    return text[: max_chars - 3] + "..."


def safe_len(value: Any) -> int:
    try:
        return len(value)  # type: ignore[arg-type]
    except Exception:
        return 0


def host_for(url: str) -> str:
    try:
        return re.sub(r"^www\\.", "", url.split("/")[2])
    except Exception:
        return "<unknown-host>"


print(f"Repo root: {REPO_ROOT}")
print(f"Pipeline dir: {PIPELINE_DIR}")
print(f"Log level: {LOG_LEVEL}")


Repo root: /Users/chetangoel/Desktop/Repositories/study-app
Pipeline dir: /Users/chetangoel/Desktop/Repositories/study-app/pipeline
Log level: INFO


In [2]:
from runner import (
    _build_crawl_audit,
    _build_sources,
    _collect_local_export_resource,
    _flatten_resources,
    _load_modules,
)
from scraper import (
    _extract_markdown_links,
    _github_raw_url,
    _http_get,
    _should_crawl_markdown_link,
    scrape_url,
)
from url_classifier import UrlType, classify
from chunker import chunk_document
from bucket_builder import build_curriculum_buckets


In [3]:
modules = _load_modules(REPO_ROOT / "build_study_data.py")
flattened, unique_entries = _flatten_resources(modules)

_type_counts = Counter(classify(url).value for url in unique_entries)

log.info(
    "Loaded curriculum: modules=%s total_refs=%s unique_urls=%s",
    len(modules), len(flattened), len(unique_entries),
)
log.info("URL type breakdown: %s", dict(sorted(_type_counts.items())))

print(f"Modules:          {len(modules)}")
print(f"Total resources:  {len(flattened)}")
print(f"Unique URLs:      {len(unique_entries)}")
print("URL types:")
for _t, _c in sorted(_type_counts.items(), key=lambda x: -x[1]):
    print(f"  {_t:<22} {_c}")


14:53:11.483 INFO content-pipeline-debug: Loaded curriculum: modules=43 total_refs=390 unique_urls=260
14:53:11.484 INFO content-pipeline-debug: URL type breakdown: {'archive': 1, 'article': 45, 'coursera': 8, 'github_markdown': 2, 'github_pdf': 2, 'github_repo': 13, 'leetcode': 159, 'platform': 3, 'shortlink': 1, 'youtube_playlist': 4, 'youtube_video': 22}


Modules:          43
Total resources:  390
Unique URLs:      260
URL types:
  leetcode               159
  article                45
  youtube_video          22
  github_repo            13
  coursera               8
  youtube_playlist       4
  platform               3
  github_markdown        2
  github_pdf             2
  shortlink              1
  archive                1


## Stage 1: Scrape + Crawl Audit

Set `MAX_URLS` to keep debugging loops fast. Start small (20-50), then increase.


In [ ]:
# MAX_URLS = 40  # Set to None for full run.
MAX_URLS = None
PRINT_EVERY = 5
PREVIEW_SEGMENT_CHARS = 260

# Skip YouTube for now (both direct resources and links discovered in GitHub markdown).
# This keeps Stage 1 loops fast and avoids long transcript fetch stalls.
MARKDOWN_CRAWL_ALLOW_HOST_SUBSTRINGS = ("leetcode.com",)

from config import pipeline_url_excluded_from_stage1_seeds


def _is_youtube_host(url: str) -> bool:
    host = urlparse(url).netloc.casefold().removeprefix("www.")
    return host in {"youtube.com", "youtu.be"}


def _keep_url_for_stage1(url: str) -> bool:
    if pipeline_url_excluded_from_stage1_seeds(url):
        return False
    # YouTube pages can be classified as "article" (channels/users). Still skip.
    if _is_youtube_host(url):
        return False

    t = classify(url)
    if t in {UrlType.YOUTUBE_VIDEO, UrlType.YOUTUBE_PLAYLIST}:
        return False
    if t in {UrlType.GITHUB_MARKDOWN, UrlType.GITHUB_PDF, UrlType.GITHUB_REPO}:
        return True
    if t == UrlType.LEETCODE:
        return True
    return False


selected_all = list(unique_entries.items())
selected = [(url, entry) for (url, entry) in selected_all if _keep_url_for_stage1(url)]
excluded = [(url, entry) for (url, entry) in selected_all if not _keep_url_for_stage1(url)]

stage1_manifest = {
    "include_policy": "github + leetcode_problems (no youtube); minus PIPELINE_STAGE1_EXCLUDED_URL_SUBSTRINGS",
    "included_types": sorted({classify(url).value for (url, _) in selected}),
    "excluded_types": sorted({classify(url).value for (url, _) in excluded}),
    "included": [
        {
            "url": url,
            "url_type": classify(url).value,
            "module_id": entry.get("module_id"),
            "module_title": entry.get("module_title"),
            "label": (entry.get("resource") or {}).get("label"),
        }
        for (url, entry) in selected
    ],
    "excluded": [
        {
            "url": url,
            "url_type": classify(url).value,
            "module_id": entry.get("module_id"),
            "module_title": entry.get("module_title"),
            "label": (entry.get("resource") or {}).get("label"),
        }
        for (url, entry) in excluded
    ],
}

manifest_path = REPO_ROOT / "output" / "stage1-github-leetcode-manifest.json"
manifest_path.parent.mkdir(parents=True, exist_ok=True)
manifest_path.write_text(json.dumps(stage1_manifest, ensure_ascii=False, indent=2), encoding="utf-8")

_kept_types = Counter(classify(url).value for (url, _) in selected)
_excl_types = Counter(classify(url).value for (url, _) in excluded)
log.info(
    "Stage 1 filter: kept=%s excluded=%s kept_by_type=%s excl_by_type=%s manifest=%s",
    len(selected),
    len(excluded),
    dict(sorted(_kept_types.items())),
    dict(sorted(_excl_types.items())),
    manifest_path,
)

# ---- Stage 1A: Collect URLs first (parents, then expanded links) ----
# This writes two artifacts so you can inspect inputs before ingestion.
STOP_AFTER_PLAN = False

parents: list[dict] = []
expanded_links: list[dict] = []
seen_parent_urls: set[str] = set()
seen_link_urls: set[str] = set()

for seed_url, seed_entry in selected:
    resource = seed_entry.get("resource") or {}

    if seed_url not in seen_parent_urls:
        seen_parent_urls.add(seed_url)
        parents.append(
            {
                "url": seed_url,
                "url_type": classify(seed_url).value,
                "module_id": seed_entry.get("module_id"),
                "module_title": seed_entry.get("module_title"),
                "module_phase": seed_entry.get("module_phase"),
                "label": resource.get("label"),
            }
        )

    if classify(seed_url) != UrlType.GITHUB_MARKDOWN:
        continue

    log.debug("Stage 1A expanding markdown links: url=%s", seed_url)
    try:
        raw_url = _github_raw_url(seed_url)
        response = _http_get(raw_url)
        response.raise_for_status()
        markdown_text = response.text
    except Exception as exc:
        log.warning("Stage 1A markdown fetch failed: url=%s error=%s", seed_url, exc)
        expanded_links.append(
            {
                "url": None,
                "url_type": None,
                "module_id": seed_entry.get("module_id"),
                "module_title": seed_entry.get("module_title"),
                "module_phase": seed_entry.get("module_phase"),
                "label": resource.get("label"),
                "parent_url": seed_url,
                "reason": f"markdown_fetch_failed:{type(exc).__name__}",
            }
        )
        continue

    for link in _extract_markdown_links(markdown_text, seed_url):
        # Skip YouTube links discovered inside markdown during debug loops.
        if _is_youtube_host(link) or classify(link) in {UrlType.YOUTUBE_VIDEO, UrlType.YOUTUBE_PLAYLIST}:
            continue
        if not _should_crawl_markdown_link(link):
            continue
        if link in seen_link_urls:
            continue
        seen_link_urls.add(link)
        expanded_links.append(
            {
                "url": link,
                "url_type": classify(link).value,
                "module_id": seed_entry.get("module_id"),
                "module_title": seed_entry.get("module_title"),
                "module_phase": seed_entry.get("module_phase"),
                "label": (resource.get("label") or "") + " (linked)",
                "parent_url": seed_url,
                "reason": "markdown_link",
            }
        )

_md_expanded = [l for l in expanded_links if l.get("reason") == "markdown_link"]
_md_failed = [l for l in expanded_links if l.get("reason", "").startswith("markdown_fetch_failed")]
log.info(
    "Stage 1A markdown expansion done: markdown_seeds=%s links_found=%s fetch_failures=%s",
    sum(1 for (u, _) in selected if classify(u) == UrlType.GITHUB_MARKDOWN),
    len(_md_expanded),
    len(_md_failed),
)
if _md_failed:
    for _f in _md_failed:
        log.warning("  markdown_fetch_failed: module=%s parent=%s reason=%s",
                    _f.get("module_id"), _f.get("parent_url"), _f.get("reason"))

parents_path = REPO_ROOT / "output" / "stage1-parents.json"
links_path = REPO_ROOT / "output" / "stage1-expanded-links.json"
plan_path = REPO_ROOT / "output" / "stage1-crawl-plan.json"
for p in (parents_path, links_path, plan_path):
    p.parent.mkdir(parents=True, exist_ok=True)

parents_path.write_text(json.dumps({"parents": parents}, ensure_ascii=False, indent=2), encoding="utf-8")
links_path.write_text(json.dumps({"links": expanded_links}, ensure_ascii=False, indent=2), encoding="utf-8")
plan_path.write_text(
    json.dumps({"parents": parents, "links": expanded_links}, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

_plan_type_counts = Counter(i.get("url_type") for i in (parents + [l for l in expanded_links if l.get("url")]))
log.info(
    "Stage 1A plan ready: parents=%s links=%s total_to_scrape=%s by_type=%s",
    len(parents),
    len([l for l in expanded_links if l.get("url")]),
    len(parents) + len([l for l in expanded_links if l.get("url")]),
    dict(sorted(_plan_type_counts.items())),
)
log.info("Stage 1A artifacts: parents=%s  links=%s  plan=%s", parents_path, links_path, plan_path)

if STOP_AFTER_PLAN:
    log.info("STOP_AFTER_PLAN=True — skipping scrape. Set STOP_AFTER_PLAN=False to run Stage 1B.")
    raise SystemExit(f"Stage 1A only. Inspect: {parents_path} and {links_path}")

# ---- Stage 1B: Execute scraping from the plan ----
# (If STOP_AFTER_PLAN=False, we scrape exactly the URLs in the plan file.)
selected = []
for item in (parents + [l for l in expanded_links if l.get("url")]):
    url = str(item["url"])

    # Enforce "no YouTube" even if the plan/checkpoint contains it.
    if _is_youtube_host(url) or classify(url) in {UrlType.YOUTUBE_VIDEO, UrlType.YOUTUBE_PLAYLIST}:
        continue

    entry = {
        "module_id": item.get("module_id"),
        "module_title": item.get("module_title"),
        "module_phase": item.get("module_phase"),
        "resource": {"label": item.get("label") or url},
    }
    selected.append((url, entry))

if MAX_URLS is not None:
    selected = selected[:MAX_URLS]

log.info("Stage 1 starting: urls=%s", len(selected))

# --- Resume support ---
# Writes a checkpoint after each URL so you can interrupt/re-run without losing progress.
# Also records the "in-progress" URL so an interrupt mid-URL doesn't cause repeated work.
checkpoint_path = REPO_ROOT / "output" / "stage1-checkpoint.json"
state_path = REPO_ROOT / "output" / "stage1-state.json"
checkpoint_path.parent.mkdir(parents=True, exist_ok=True)

collected_resources: list[dict] = []
if checkpoint_path.exists():
    try:
        loaded = json.loads(checkpoint_path.read_text(encoding="utf-8"))
        if isinstance(loaded, list):
            collected_resources = [r for r in loaded if isinstance(r, dict)]
    except Exception:
        collected_resources = []

processed_urls = {
    str(r.get("resource_url") or "")
    for r in collected_resources
    if str(r.get("resource_url") or "")
}

# If the previous run died mid-URL, record it once as interrupted so we can move on.
if state_path.exists():
    try:
        state = json.loads(state_path.read_text(encoding="utf-8"))
    except Exception:
        state = {}
    in_progress_url = str((state or {}).get("in_progress_url") or "")
    if in_progress_url and in_progress_url not in processed_urls:
        collected_resources.append(
            {
                "module_id": state.get("module_id"),
                "module_title": state.get("module_title"),
                "module_phase": state.get("module_phase"),
                "label": state.get("label"),
                "resource_url": in_progress_url,
                "resource_type": "interrupted",
                "resolved_url": None,
                "title": None,
                "description": None,
                "status": "failed",
                "error": "interrupted: previous run ended mid-scrape",
                "segments": [],
                "_debug": {"last_seen_at": state.get("updated_at")},
            }
        )
        processed_urls.add(in_progress_url)
        checkpoint_path.write_text(json.dumps(collected_resources, ensure_ascii=False, indent=2), encoding="utf-8")

log.info(
    "Stage 1 resume: checkpoint=%s processed=%s remaining=%s",
    checkpoint_path,
    len(processed_urls),
    max(len(selected) - len(processed_urls), 0),
)

status_counter: Counter[str] = Counter()
host_counter: Counter[str] = Counter()
for r in collected_resources:
    if not isinstance(r, dict):
        continue
    status_counter[str(r.get("status") or "")] += 1
    res_url = str(r.get("resource_url") or "")
    if res_url:
        host_counter[host_for(res_url)] += 1

stage_timer = Timer("stage1")
for i, (url, entry) in enumerate(selected, start=1):
    # Defensive: in case anything slipped into `selected`, never scrape YouTube here.
    if _is_youtube_host(url) or classify(url) in {UrlType.YOUTUBE_VIDEO, UrlType.YOUTUBE_PLAYLIST}:
        processed_urls.add(url)
        if i == 1 or i % PRINT_EVERY == 0 or i == len(selected):
            log.info("[%s/%s] SKIP youtube url=%s", i, len(selected), url)
        continue

    if url in processed_urls:
        if i == 1 or i % PRINT_EVERY == 0 or i == len(selected):
            log.info("[%s/%s] SKIP already-processed url=%s", i, len(selected), url)
        continue

    t = Timer(f"scrape:{i}")
    resource = entry["resource"]
    module_id = entry.get("module_id")
    label = resource.get("label")
    host = host_for(url)
    host_counter[host] += 1

    # Record which URL is currently in-flight (helps resume after interrupt mid-URL).
    state_path.write_text(
        json.dumps(
            {
                "in_progress_url": url,
                "i": i,
                "total": len(selected),
                "module_id": module_id,
                "module_title": entry.get("module_title"),
                "module_phase": entry.get("module_phase"),
                "label": label,
                "updated_at": time.time(),
            },
            ensure_ascii=False,
            indent=2
        ),
        encoding="utf-8",
    )

    try:
        if resource.get("sourcePath"):
            log.debug("[%s/%s] LOCAL start module=%s host=%s label=%s", i, len(selected), module_id, host, label)
            collected = _collect_local_export_resource(entry)
            status = str(collected.get("status") or "local")
            segments = collected.get("segments") or []
            status_counter[status] += 1
            log.debug(
                "[%s/%s] LOCAL done module=%s host=%s label=%s status=%s segments=%s elapsed=%sms",
                i,
                len(selected),
                module_id,
                host,
                label,
                status,
                safe_len(segments),
                t.ms(),
            )
        else:
            log.info("[%s/%s] SCRAPE start module=%s host=%s url=%s", i, len(selected), module_id, host, url)
            scrape_result = scrape_url(url)
            segments = scrape_result.segments or []
            status = str(scrape_result.status)
            status_counter[status] += 1
            log.info(
                "[%s/%s] SCRAPE done module=%s host=%s status=%s type=%s segs=%s elapsed=%sms",
                i,
                len(selected),
                module_id,
                host,
                status,
                scrape_result.url_type.value,
                safe_len(segments),
                t.ms(),
            )

            seg_chars = [len(str(s.get("text", ""))) for s in segments if isinstance(s, dict)]
            seg_total = sum(seg_chars)
            seg_preview = ""
            if segments:
                first = segments[0]
                if isinstance(first, dict):
                    seg_preview = preview_text(first.get("text") or "", max_chars=PREVIEW_SEGMENT_CHARS)
                else:
                    seg_preview = preview_text(first, max_chars=PREVIEW_SEGMENT_CHARS)

            collected = {
                "module_id": entry["module_id"],
                "module_title": entry["module_title"],
                "module_phase": entry["module_phase"],
                "label": resource["label"],
                "resource_url": url,
                "resource_type": scrape_result.url_type.value,
                "resolved_url": scrape_result.resolved_url,
                "title": scrape_result.title,
                "description": scrape_result.og_description,
                "status": scrape_result.status,
                "error": scrape_result.error,
                "segments": scrape_result.segments,
                "_debug": {
                    "elapsed_ms": t.ms(),
                    "segments": safe_len(segments),
                    "segment_chars_total": seg_total,
                    "segment_chars_min": min(seg_chars) if seg_chars else 0,
                    "segment_chars_max": max(seg_chars) if seg_chars else 0,
                    "segment_preview": seg_preview,
                },
            }

            level = logging.INFO
            if status in {"failed", "fallback"}:
                level = logging.WARNING
            elif status not in {"ok", "success"}:
                level = logging.DEBUG

            log.log(
                level,
                "[%s/%s] %s host=%s status=%s type=%s segs=%s chars=%s title=%s err=%s elapsed=%sms",
                i,
                len(selected),
                module_id,
                host,
                status,
                scrape_result.url_type.value,
                safe_len(segments),
                seg_total,
                preview_text(scrape_result.title),
                preview_text(scrape_result.error),
                t.ms(),
            )
            if status in {"failed", "fallback"} and seg_preview:
                log.warning("  preview: %s", seg_preview)

    except Exception as e:
        status_counter["exception"] += 1
        collected = {
            "module_id": entry.get("module_id"),
            "module_title": entry.get("module_title"),
            "module_phase": entry.get("module_phase"),
            "label": resource.get("label"),
            "resource_url": url,
            "resource_type": "exception",
            "resolved_url": None,
            "title": None,
            "description": None,
            "status": "failed",
            "error": f"exception: {type(e).__name__}: {e}",
            "segments": [],
            "_debug": {"elapsed_ms": t.ms()},
        }
        log.exception("[%s/%s] EXCEPTION %s url=%s", i, len(selected), module_id, url)

    collected_resources.append(collected)
    processed_urls.add(url)

    # Persist progress so Stage 1 can be resumed after interruptions.
    checkpoint_path.write_text(json.dumps(collected_resources, ensure_ascii=False, indent=2), encoding="utf-8")
    state_path.write_text(json.dumps({"in_progress_url": None, "updated_at": time.time()}, ensure_ascii=False, indent=2), encoding="utf-8")

    if i % PRINT_EVERY == 0 or i == len(selected):
        log.info(
            "Progress: %s/%s done (stage elapsed=%ss) status_counts=%s checkpoint=%s",
            i,
            len(selected),
            round(stage_timer.ms() / 1000, 1),
            dict(status_counter),
            checkpoint_path,
        )

log.info(
    "Stage 1 done: resources=%s elapsed=%ss top_hosts=%s",
    len(collected_resources),
    round(stage_timer.ms() / 1000, 1),
    host_counter.most_common(8),
)

audit = _build_crawl_audit(collected_resources)
audit["status_counts"]


14:53:11.512 INFO content-pipeline-debug: Stage 1 filter: kept=176 excluded=84 kept_by_type={'github_markdown': 2, 'github_pdf': 2, 'github_repo': 13, 'leetcode': 159} excl_by_type={'archive': 1, 'article': 45, 'coursera': 8, 'platform': 3, 'shortlink': 1, 'youtube_playlist': 4, 'youtube_video': 22} manifest=/Users/chetangoel/Desktop/Repositories/study-app/output/stage1-github-leetcode-manifest.json
14:53:11.929 INFO content-pipeline-debug: Stage 1A markdown expansion done: markdown_seeds=2 links_found=6 fetch_failures=0
14:53:11.932 INFO content-pipeline-debug: Stage 1A plan ready: parents=176 links=6 total_to_scrape=182 by_type={'github_markdown': 2, 'github_pdf': 2, 'github_repo': 13, 'leetcode': 165}
14:53:11.932 INFO content-pipeline-debug: Stage 1A artifacts: parents=/Users/chetangoel/Desktop/Repositories/study-app/output/stage1-parents.json  links=/Users/chetangoel/Desktop/Repositories/study-app/output/stage1-expanded-links.json  plan=/Users/chetangoel/Desktop/Repositories/study

{'ok': 23, 'fallback': 190, 'failed': 4}

In [5]:
# If you run this cell without re-running Stage 1,
# rebuild/load the audit from artifacts.
if "audit" not in globals():
    if "collected_resources" in globals() and collected_resources:
        audit = _build_crawl_audit(collected_resources)
    else:
        # Fall back to the Stage 1 checkpoint (list of collected resource rows).
        checkpoint_guess = REPO_ROOT / "output" / "stage1-checkpoint.json"
        if not checkpoint_guess.exists():
            raise RuntimeError(
                "Missing `audit` and no Stage 1 artifacts found. "
                "Run Stage 1 first, or ensure output/stage1-checkpoint.json exists."
            )
        collected_resources = json.loads(checkpoint_guess.read_text(encoding="utf-8"))
        audit = _build_crawl_audit(collected_resources)


degraded = [
    r
    for r in audit.get("degraded_resources", [])
    if r.get("status") in {"fallback", "failed"}
]
ok_count = int(audit.get("total_resources", 0)) - len(degraded)
fallback_count = sum(1 for r in degraded if r.get("status") == "fallback")
failed_count = sum(1 for r in degraded if r.get("status") == "failed")

log.info(
    "Stage 1 results: total=%s  ok=%s  fallback=%s  failed=%s",
    audit.get("total_resources"),
    ok_count,
    fallback_count,
    failed_count,
)

by_module = Counter(r.get("module_id") for r in degraded if r.get("module_id"))


def _host_for_row(row: dict) -> str:
    url = (row.get("resource_url") or "").strip()
    parsed = urlparse(url)

    # Handle bare hosts / schemeless URLs.
    if parsed.netloc:
        host = parsed.netloc
    else:
        # urlparse('example.com/x') puts it in path.
        host = url.split("/")[0]

    host = re.sub(r"^www\.", "", host)
    return host or "<unknown>"


by_host = Counter(_host_for_row(r) for r in degraded)
by_status = Counter(r.get("status") for r in degraded if r.get("status"))

log.info("Degraded by status:  %s", dict(sorted(by_status.items())))
log.info("Degraded by host:    %s", dict(by_host.most_common(10)))
log.info("Degraded by module:  %s", dict(by_module.most_common(10)))

if failed_count:
    log.warning("%s URLs with status=failed (network or scraper errors)", failed_count)

print(f"Total:    {audit.get('total_resources')}")
print(f"OK:       {ok_count}")
print(f"Fallback: {fallback_count}")
print(f"Failed:   {failed_count}")
print("\nTop degraded modules:")
for module_id, count in by_module.most_common(10):
    print(f"  {module_id}: {count}")
print("\nTop degraded hosts:")
for host, count in by_host.most_common(10):
    print(f"  {host}: {count}")


14:53:11.966 INFO content-pipeline-debug: Stage 1 results: total=217  ok=23  fallback=190  failed=4
14:53:11.967 INFO content-pipeline-debug: Degraded by status:  {'failed': 4, 'fallback': 190}
14:53:11.967 INFO content-pipeline-debug: Degraded by host:    {'leetcode.com': 160, 'youtube.com': 28, 'youtu.be': 5, 'github.com': 1}
14:53:11.968 INFO content-pipeline-debug: Degraded by module:  {'leetcode-course-traversals-trees-graphs': 29, 'leetcode-course-hashing': 17, 'leetcode-course-arraystrings': 15, 'leetcode-course-dynamic-programming': 15, 'leetcode-course-stacks-and-queues': 12, 'leetcode-export-bonus': 12, 'leetcode-course-heaps': 11, 'leetcode-course-binary-search': 11, 'leetcode-course-backtracking': 11, 'leetcode-course-linked-lists': 9}
14:53:11.968 WARNING content-pipeline-debug: 4 URLs with status=failed (network or scraper errors)


Total:    217
OK:       23
Fallback: 190
Failed:   4

Top degraded modules:
  leetcode-course-traversals-trees-graphs: 29
  leetcode-course-hashing: 17
  leetcode-course-arraystrings: 15
  leetcode-course-dynamic-programming: 15
  leetcode-course-stacks-and-queues: 12
  leetcode-export-bonus: 12
  leetcode-course-heaps: 11
  leetcode-course-binary-search: 11
  leetcode-course-backtracking: 11
  leetcode-course-linked-lists: 9

Top degraded hosts:
  leetcode.com: 160
  youtube.com: 28
  youtu.be: 5
  github.com: 1


In [6]:
failed_rows = [r for r in degraded if r["status"] == "failed"]
fallback_rows = [r for r in degraded if r["status"] == "fallback"]
failed_error_counts = Counter((r.get("error") or "") for r in failed_rows)
fallback_error_counts = Counter((r.get("error") or "") for r in fallback_rows)

log.info("Failed error signatures (%s total):", len(failed_rows))
for error, count in failed_error_counts.most_common(10):
    short = error if len(error) < 140 else error[:137] + "..."
    log.info("  %3s  %s", count, short)

log.info("Fallback reasons (%s total):", len(fallback_rows))
for error, count in fallback_error_counts.most_common(10):
    short = error if len(error) < 140 else error[:137] + "..."
    log.info("  %3s  %s", count, short)

# Spotlight: any bot-challenge fallbacks (Cloudflare etc.)
bot_blocked = [r for r in fallback_rows if "bot-challenge" in (r.get("error") or "")]
if bot_blocked:
    log.warning("%s URL(s) blocked by bot-protection:", len(bot_blocked))
    for r in bot_blocked[:10]:
        log.warning("  [%s] %s", r.get("module_id"), r.get("resource_url"))

print("Top failed error signatures:")
for error, count in failed_error_counts.most_common(10):
    short = error if len(error) < 160 else error[:157] + "..."
    print(f"  {count:>3}  {short}")
print("\nTop fallback reasons:")
for error, count in fallback_error_counts.most_common(10):
    short = error if len(error) < 160 else error[:157] + "..."
    print(f"  {count:>3}  {short}")


14:53:11.974 INFO content-pipeline-debug: Failed error signatures (4 total):
14:53:11.975 INFO content-pipeline-debug:     3  interrupted: previous run ended mid-scrape
14:53:11.975 INFO content-pipeline-debug:     1  Invalid IPv6 URL
14:53:11.976 INFO content-pipeline-debug: Fallback reasons (190 total):
14:53:11.976 INFO content-pipeline-debug:   189  
14:53:11.976 INFO content-pipeline-debug:     1  skipped: bare-domain landing page


Top failed error signatures:
    3  interrupted: previous run ended mid-scrape
    1  Invalid IPv6 URL

Top fallback reasons:
  189  
    1  skipped: bare-domain landing page


## Stage 2: Build Sources + Chunks


In [7]:
CHUNK_PREVIEW_CHARS = 240

stage_timer = Timer("stage2")
sources = _build_sources(collected_resources)

_src_type_counts = Counter(s.get("resource_type", "unknown") for s in sources)
_src_char_counts = Counter()
for _s in sources:
    _src_char_counts[_s.get('resource_type', 'unknown')] += len(str(_s.get('text') or ''))
log.info(
    "Stage 2 starting: sources=%s by_type=%s",
    len(sources),
    dict(sorted(_src_type_counts.items())),
)
log.info(
    "Stage 2 source chars by type: %s",
    {k: f"{v:,}" for k, v in sorted(_src_char_counts.items(), key=lambda x: -x[1])},
)

chunks = []
chunk_counts_by_module: Counter[str] = Counter()
chunk_counts_by_host: Counter[str] = Counter()

for idx, source in enumerate(sources, start=1):
    t = Timer(f"chunk:{idx}")
    source_url = str(source.get("resource_url") or "")
    module_id = str(source.get("module_id") or "<unknown-module>")
    host = host_for(source_url) if source_url else "<no-url>"

    source_chunks = chunk_document(source)
    chunks.extend(source_chunks)

    chunk_counts_by_module[module_id] += len(source_chunks)
    chunk_counts_by_host[host] += len(source_chunks)

    source_text = str(source.get("text") or "")
    _log_fn = log.info if (idx % 20 == 0 or idx == 1 or idx == len(sources)) else log.debug
    _log_fn(
        "[%s/%s] chunk module=%s type=%s host=%s text_chars=%s chunks=%s elapsed=%sms",
        idx,
        len(sources),
        module_id,
        str(sources[idx - 1].get('resource_type') or 'unknown'),
        host,
        len(source_text),
        len(source_chunks),
        t.ms(),
    )

    if idx % 20 == 0 or idx == len(sources):
        log.info(
            "Chunk progress: %s/%s sources (elapsed=%ss, total_chunks=%s)",
            idx,
            len(sources),
            round(stage_timer.ms() / 1000, 1),
            len(chunks),
        )

print(f"Source docs: {len(sources)}")
print(f"Chunks: {len(chunks)}")

chunk_texts = [str(c.get("text", "")) for c in chunks if isinstance(c, dict)]
chunk_lengths = [len(t) for t in chunk_texts]
if chunk_lengths:
    print(
        "Chunk length min/avg/max:",
        min(chunk_lengths),
        sum(chunk_lengths) // len(chunk_lengths),
        max(chunk_lengths),
    )

    # Preview a few chunks to sanity-check chunk boundaries/content.
    indexed = list(enumerate(chunk_texts))
    indexed.sort(key=lambda it: len(it[1]))

    smallest = indexed[:2]
    largest = indexed[-2:]

    print("\nChunk previews (smallest):")
    for i, text in smallest:
        print(textwrap.fill(preview_text(text, max_chars=CHUNK_PREVIEW_CHARS), width=100))
        print("-")

    print("Chunk previews (largest):")
    for i, text in largest:
        print(textwrap.fill(preview_text(text, max_chars=CHUNK_PREVIEW_CHARS), width=100))
        print("-")

print("\nTop modules by chunk count:")
for module_id, count in chunk_counts_by_module.most_common(10):
    print(f"  {module_id}: {count}")

print("\nTop hosts by chunk count:")
for host, count in chunk_counts_by_host.most_common(10):
    print(f"  {host}: {count}")

_chunk_type_counts = Counter(str(c.get('resource_type') or c.get('kind') or 'unknown') for c in chunks if isinstance(c, dict))
log.info(
    "Stage 2 done: chunks=%s elapsed=%ss chunks_by_type=%s top_modules=%s",
    len(chunks),
    round(stage_timer.ms() / 1000, 1),
    dict(sorted(_chunk_type_counts.items())),
    dict(chunk_counts_by_module.most_common(5)),
)


14:53:11.986 INFO content-pipeline-debug: Stage 2 starting: sources=45 by_type={'article': 15, 'github_markdown': 10, 'github_pdf': 2, 'github_repo': 14, 'leetcode': 4}
14:53:11.987 INFO content-pipeline-debug: Stage 2 source chars by type: {'github_repo': '76,437', 'article': '22,314', 'github_markdown': '16,630', 'leetcode': '3,974', 'github_pdf': '2,113'}
14:53:11.988 INFO content-pipeline-debug: [1/45] chunk module=search-bitwise type=github_pdf host=github.com text_chars=1272 chunks=1 elapsed=0ms
14:53:11.989 INFO content-pipeline-debug: [20/45] chunk module=setup-habits type=article host=youtube.com text_chars=164 chunks=1 elapsed=0ms
14:53:11.989 INFO content-pipeline-debug: Chunk progress: 20/45 sources (elapsed=0.0s, total_chunks=31)
14:53:11.991 INFO content-pipeline-debug: [40/45] chunk module=supplemental-ml-interviews type=github_repo host=github.com text_chars=5999 chunks=2 elapsed=0ms
14:53:11.991 INFO content-pipeline-debug: Chunk progress: 40/45 sources (elapsed=0.0s, 

Source docs: 45
Chunks: 133
Chunk length min/avg/max: 5 888 3500

Chunk previews (smallest):
| No.
-
A static IP ad
-
Chunk previews (largest):
Everything you need to create on YouTube No matter what kind of information, advice, or help youâre
looking for, this is the spot. Top questions Creators have questions and weâve got answers. We
compiled the most common questions we get ...
-
- C++ - [C++ Cheat Sheet](https://github.com/jwasham/coding-interview-
university/blob/main/extras/cheat%20sheets/Cpp_reference.pdf) - [STL Cheat
Sheet](https://github.com/jwasham/coding-interview-university/blob/main/extras/cheat%20sheet...
-

Top modules by chunk count:
  setup-habits: 46
  supplemental-interview-handbooks: 31
  supplemental-system-design-interviews: 19
  supplemental-frontend-interviews: 11
  system-design: 11
  supplemental-ml-interviews: 10
  supplemental-behavioral-interviews: 4
  search-bitwise: 1

Top hosts by chunk count:
  github.com: 111
  www.youtube.com: 12
  youtube.com: 6
  

## Stage 3: Bucket Assignment Diagnostics


In [8]:
stage_timer = Timer("stage3")
log.info("Stage 3 starting: chunks=%s", len(chunks))
buckets, bucket_diagnostics = build_curriculum_buckets(chunks)
log.info("Stage 3 done: buckets=%s elapsed=%ss", len(buckets), round(stage_timer.ms() / 1000, 1))

print(bucket_diagnostics)

# ----- Coverage + evidence diagnostics -----

def _chunk_id(c: dict[str, Any]) -> str:
    # Try common keys; fall back to object identity.
    for key in ("chunk_id", "id", "hash"):
        v = c.get(key)
        if v:
            return str(v)
    return str(id(c))


all_chunk_ids = {_chunk_id(c) for c in chunks if isinstance(c, dict)}
assigned_chunk_ids: set[str] = set()

topic_evidence: Counter[str] = Counter()

print("\nBucket evidence summary:")
for bucket in buckets:
    general_chunks = bucket.get("general_chunks") or []
    topic_matches = bucket.get("topic_matches") or {}
    topic_ids = bucket.get("topic_ids") or []

    topic_match_count = 0
    topic_evidence_chunks: set[str] = set()

    for topic_id in topic_ids:
        matches = topic_matches.get(topic_id) or []
        topic_match_count += safe_len(matches)
        topic_evidence[topic_id] += safe_len(matches)
        for m in matches:
            if isinstance(m, dict):
                topic_evidence_chunks.add(_chunk_id(m))

    for c in general_chunks:
        if isinstance(c, dict):
            assigned_chunk_ids.add(_chunk_id(c))
    assigned_chunk_ids |= topic_evidence_chunks

    bucket_total = safe_len(general_chunks) + len(topic_evidence_chunks)
    pct = (bucket_total / max(len(all_chunk_ids), 1)) * 100

    print(
        f"  {bucket.get('bucket_id')}: total≈{bucket_total} ({pct:.1f}%), "
        f"general={safe_len(general_chunks)}, topic-match-rows={topic_match_count}, "
        f"topic-evidence-chunks={len(topic_evidence_chunks)}, source-urls={safe_len(bucket.get('source_urls') or [])}"
    )

unassigned = len(all_chunk_ids - assigned_chunk_ids)
assigned = len(assigned_chunk_ids)
print("\nCoverage:")
print(
    {
        "total_chunks": len(all_chunk_ids),
        "assigned_chunks": assigned,
        "unassigned_chunks": unassigned,
        "assigned_pct": round(assigned / max(len(all_chunk_ids), 1) * 100, 1),
    }
)

log.info(
    "Stage 3 coverage: total_chunks=%s assigned=%s unassigned=%s assigned_pct=%s%%",
    len(all_chunk_ids), assigned, unassigned,
    round(assigned / max(len(all_chunk_ids), 1) * 100, 1),
)

_zero_buckets = [b.get("bucket_id") for b, sz in zip(buckets, [s for _, s in size_by_bucket]) if sz == 0]
if _zero_buckets:
    log.warning("Buckets with zero chunks (%s): %s", len(_zero_buckets), _zero_buckets)
else:
    log.info("All buckets have at least one chunk")

print("\nTop topics by evidence rows:")
for topic_id, count in topic_evidence.most_common(25):
    print(f"  {topic_id}: {count}")

# Helpful to spot buckets that are huge / empty.
size_by_bucket = []
for bucket in buckets:
    general_chunks = bucket.get("general_chunks") or []
    topic_matches = bucket.get("topic_matches") or {}
    topic_ids = bucket.get("topic_ids") or []
    evidence_chunks: set[str] = set()
    for topic_id in topic_ids:
        for m in (topic_matches.get(topic_id) or []):
            if isinstance(m, dict):
                evidence_chunks.add(_chunk_id(m))
    size_by_bucket.append((bucket.get("bucket_id"), safe_len(general_chunks) + len(evidence_chunks)))

size_by_bucket.sort(key=lambda it: it[1], reverse=True)
print("\nLargest buckets:")
for bucket_id, size in size_by_bucket[:10]:
    print(f"  {bucket_id}: {size}")

print("\nSmallest buckets:")
for bucket_id, size in size_by_bucket[-10:]:
    print(f"  {bucket_id}: {size}")


14:53:12.006 INFO content-pipeline-debug: Stage 3 starting: chunks=133
14:53:12.093 INFO content-pipeline-debug: Stage 3 done: buckets=9 elapsed=0.1s
14:53:12.094 INFO content-pipeline-debug: Stage 3 coverage: total_chunks=133 assigned=180 unassigned=94 assigned_pct=135.3%


{'total_chunks': 133, 'assigned_chunks': 133, 'unassigned_chunks': 0, 'bucket_chunk_counts': {'foundations-and-analysis': 55, 'linear-structures-and-patterns': 11, 'trees-search-and-ordering': 19, 'recursive-and-optimization-paradigms': 8, 'graph-algorithms-and-traversal': 1, 'oop-and-architecture': 1, 'distributed-systems-and-platforms': 0, 'system-design-curriculum': 41, 'interview-preparation-and-career': 48}, 'chunk_diagnostics': [{'chunk_id': 'chunk:51a698785b9b', 'title': 'Bits Cheat Sheet', 'module_id': 'search-bitwise', 'matched_topic_ids': [], 'matched_bucket_ids': ['linear-structures-and-patterns', 'trees-search-and-ordering']}, {'chunk_id': 'chunk:cb61d0d9620e', 'title': 'CIU Flashcards Repo', 'module_id': 'setup-habits', 'matched_topic_ids': ['active-recall-and-spaced-repetition'], 'matched_bucket_ids': ['foundations-and-analysis']}, {'chunk_id': 'chunk:cc2776fb6830', 'title': 'CIU Flashcards Repo', 'module_id': 'setup-habits', 'matched_topic_ids': ['active-recall-and-space

NameError: name 'size_by_bucket' is not defined

## Stage 4 (Optional): LLM Synthesis + Validation

This stage needs `LLM_API_KEY` or `OPENROUTER_API_KEY` and is slower.


In [ ]:
RUN_LLM_STAGES = False

if RUN_LLM_STAGES:
    from bucket_synthesizer import synthesize_bucket_topics
    from topic_lesson_synthesizer import synthesize_topic_lessons
    from topic_validator import canonicalize_topics, validate_topic_lessons
    from edge_builder import build_topic_edges
    from planning_graph import build_planning_graph

    if not (os.getenv("LLM_API_KEY") or os.getenv("OPENROUTER_API_KEY")):
        raise RuntimeError("Set LLM_API_KEY or OPENROUTER_API_KEY before RUN_LLM_STAGES=True")

    synthesized_topics = synthesize_bucket_topics(buckets)
    canonical_topics, canonical_diag = canonicalize_topics(synthesized_topics)
    topics_with_lessons, lesson_synth_diag = synthesize_topic_lessons(canonical_topics, buckets)
    validated_topics, lesson_validation_diag = validate_topic_lessons(topics_with_lessons)
    topic_edges, edge_diag = build_topic_edges(validated_topics, include_diagnostics=True)
    planning_graph = build_planning_graph(validated_topics)

    print({
        "input_topics": len(synthesized_topics),
        "canonical_topics": len(canonical_topics),
        "validated_topics": len(validated_topics),
        "topic_edges": len(topic_edges),
        "planning_topics": planning_graph["planning_validation"]["covered_planning_topics"],
        "merged_topics": canonical_diag.get("merged_topic_count", 0),
        "invalid_lessons": lesson_validation_diag.get("invalid_topic_count", 0),
    })
else:
    print("Skipping Stage 4. Set RUN_LLM_STAGES=True when ready.")


## Quick Quality Checks Against Existing `knowledge-base.json`

Use this to quickly inspect production artifacts without rerunning the full pipeline.


In [ ]:
kb_path = REPO_ROOT / "knowledge-base.json"
kb = json.loads(kb_path.read_text(encoding="utf-8"))

planning_topics = kb.get("planning_topics", [])
topics = kb.get("topics", [])

planning_missing_guide = sum(1 for t in planning_topics if not str(t.get("study_guide_markdown") or "").strip())
topic_missing_guide = sum(1 for t in topics if not str(t.get("study_guide_markdown") or "").strip())

_kb_stats = {
    "kb_version": kb.get("version"),
    "generated_at": kb.get("generated_at"),
    "planning_topics": len(planning_topics),
    "topics": len(topics),
    "planning_topics_missing_guide": planning_missing_guide,
    "topics_missing_guide": topic_missing_guide,
}
log.info("knowledge-base.json: %s", _kb_stats)
if planning_missing_guide:
    log.warning("%s planning topics missing study_guide_markdown", planning_missing_guide)
if topic_missing_guide:
    log.warning("%s topics missing study_guide_markdown", topic_missing_guide)
print(_kb_stats)


In [ ]:
all_source_urls = sorted({url for t in planning_topics for url in (t.get("source_urls") or [])})

noisy_patterns = [
    r"/about$",
    r"/search$",
    r"/category",
    r"/tag/",
    r"/reviews",
    r"/terms$",
    r"/privacy",
]

def likely_noisy(url: str) -> bool:
    low = url.lower()
    return any(re.search(p, low) for p in noisy_patterns)

suspect_urls = [u for u in all_source_urls if likely_noisy(u)]
log.info("KB source URLs: total=%s likely_noisy=%s", len(all_source_urls), len(suspect_urls))
if suspect_urls:
    log.warning("Suspect (noisy) source URLs in knowledge-base:")
    for _u in suspect_urls[:30]:
        log.warning("  %s", _u)

print(f"Unique source URLs: {len(all_source_urls)}")
print(f"Likely noisy URLs:  {len(suspect_urls)}")
suspect_urls[:30]
